# Evaluation 2 – Member 4 Assigned Model: Gradient Boosting

## 1. Introduction

This notebook implements **Member 4's assigned model** for Evaluation 2 of the
IT3051 Hotel Booking Cancellation Prediction group project.

- **Project Domain:** Hotel Booking Cancellation Prediction
- **Target Variable:** `is_canceled` (binary: 0 = Not Cancelled, 1 = Cancelled)
- **Assigned Model:** Boosting Algorithm (`sklearn.ensemble.GradientBoostingClassifier`)
- **Data Source:** Evaluation 1 preprocessed dataset and pipeline reused without modification

### Evaluation 1 Completed Responsibilities

| Member | Responsibility | Status |
|:---|:---|:---|
| **Member 1** | Exploratory Data Analysis & Target Analysis | Completed |
| **Member 2** | Data Quality Analysis (Missing values, Duplicates, Outliers) | Completed |
| **Member 3** | Data Leakage Investigation & Feature Engineering | Completed |
| **Member 4** | Preprocessing, Encoding, Scaling, Train/Test Split | Completed |

### Evaluation 2 Model Division

| Member | Assigned Model |
|:---|:---|
| Member 1 | Logistic Regression |
| Member 2 | Decision Tree |
| Member 3 | Random Forest |
| **Member 4** | **Gradient Boosting** *(this notebook)* |

> **Strict Scope:** This notebook implements **only** the Gradient Boosting model.
> Final group model selection happens only after all four models are compared fairly.

> **Data Leakage Safeguard:** The test set (`X_test`, `y_test`) is strictly held out during
> training and hyperparameter tuning. It is evaluated only once on the frozen final model.


## 2. Existing Preprocessing and Data Loading

We load and execute the standardized group pipeline from `src/preprocessing/preprocessing.py`
and `src/feature_engineering/feature_engineering.py` to ensure exact parity across group models.

**Pipeline Summary:**
1. Load cleaned data: 87,228 rows (after removing 31,994 duplicates and 168 invalid guest/rate entries).
2. Drop confirmed post-outcome leakage features: `reservation_status`, `reservation_status_date`.
3. Engineer 8 domain features (`total_stay`, `total_guests`, `is_family`, `room_changed`, `adr_per_guest`, etc.).
4. Partition into predictors $X$ (37 features) and target $y$ (`is_canceled`).
5. Stratified 80/20 train/test split with `random_state=42`.
6. Apply `ColumnTransformer` (StandardScaler on 25 numericals + OneHotEncoder on 12 categoricals).


In [ ]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
import matplotlib.pyplot as plt
import seaborn as sns

# Locate repository root
repo_candidates = [Path.cwd(), *Path.cwd().parents]
REPO_ROOT = next(
    (p for p in repo_candidates if (p / 'data/raw/hotel_bookings.csv').exists()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError('Cannot locate data/raw/hotel_bookings.csv. Please verify root path.')

src_path = str(REPO_ROOT / 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)

# Evaluation 1 pipeline functions
from feature_engineering.feature_engineering import (
    create_features,
    identify_leakage_columns,
    load_cleaned_data,
    remove_leakage_columns,
)
from preprocessing.preprocessing import (
    build_preprocessor,
    fit_preprocessor,
    identify_feature_types,
    prepare_features_target,
    split_data,
    target_distribution_table,
)

# Member 4 helper module
from modeling.gradient_boosting import (
    evaluate_classifier,
    plot_confusion_matrix,
    plot_roc_curve,
    comparison_table,
    overfitting_check,
    plot_feature_importance,
)

RANDOM_STATE = 42
warnings.filterwarnings('ignore')
print('All imports loaded successfully. Repository root:', REPO_ROOT)


In [ ]:
# Step 1: Load cleaned data and remove leakage columns
DATA_PATH     = REPO_ROOT / 'data/raw/hotel_bookings.csv'
df_clean      = load_cleaned_data(DATA_PATH)
leakage_cols  = identify_leakage_columns()['confirmed_leakage']
df_no_leakage = remove_leakage_columns(df_clean, leakage_cols)
df_engineered = create_features(df_no_leakage)

print(f'Engineered dataset shape: {df_engineered.shape}')
print(f'Confirmed leakage columns dropped: {leakage_cols}')


In [ ]:
# Step 2: Separate features and target
X, y, _ = prepare_features_target(df_engineered, target_col='is_canceled')
numerical_features, categorical_features = identify_feature_types(X)

print(f'X shape : {X.shape}')
print(f'y shape : {y.shape}')
print(f'Numerical features   ({len(numerical_features)})')
print(f'Categorical features ({len(categorical_features)})')
print(f'Class balance in full dataset: {y.value_counts().to_dict()}')


In [ ]:
# Step 3: Stratified 80/20 train/test split using group-wide random_state=42
X_train, X_test, y_train, y_test = split_data(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)
print('Train/test split completed successfully.')


In [ ]:
# Step 4: Fit preprocessor on X_train ONLY, then transform X_test
preprocessor = build_preprocessor(numerical_features, categorical_features)
X_train_proc, X_test_proc = fit_preprocessor(preprocessor, X_train, X_test)

# Extract clean feature names for feature importance analysis
processed_feature_names = preprocessor.get_feature_names_out()
feature_names_clean = [
    n.replace('num__', '').replace('cat__', '')
    for n in processed_feature_names
]

print(f'X_train_proc shape : {X_train_proc.shape}')
print(f'X_test_proc  shape : {X_test_proc.shape}')
print(f'Total preprocessed features (including one-hot encoding): {len(feature_names_clean)}')


## 3. Train / Test Data Verification

Verify shapes, class balance, and complete absence of target/leakage features.


In [ ]:
shapes = pd.DataFrame({
    'Dataset Object': ['X_train', 'X_test', 'y_train', 'y_test', 'X_train_proc', 'X_test_proc'],
    'Shape': [str(X_train.shape), str(X_test.shape), str(y_train.shape), str(y_test.shape),
              str(X_train_proc.shape), str(X_test_proc.shape)],
})
display(shapes)

dist = target_distribution_table(y, y_train, y_test)
print('\nTarget distribution across splits:')
display(dist)

# Defensive assertions
assert 'is_canceled' not in X.columns, 'Target is_canceled found in X!'
assert not any(col in X.columns for col in ['reservation_status', 'reservation_status_date']), 'Leakage in X!'
print('Integrity checks passed: No leakage columns and target is strictly isolated.')


## 4. Selected Boosting Algorithm: `sklearn.ensemble.GradientBoostingClassifier`

### Rationale for Selection
- **Environment Compatibility:** `scikit-learn==1.9.1` is natively installed in the project environment,
  while `xgboost` is not present in `requirements.txt`. Using `GradientBoostingClassifier` guarantees
  100% reproducibility across all machines without external package dependencies.
- **Algorithm Foundations:** `GradientBoostingClassifier` implements Friedman's Gradient Boosting Machine (GBM).
  It sequentially fits decision trees to the negative gradient (pseudo-residuals) of the binomial deviance loss.
- **Feature Importance Support:** Provides native Mean Decrease in Impurity (MDI) / deviance reduction
  via `feature_importances_`.
- **Efficiency:** With `subsample` and `max_features` regularisation, it fits rapidly and avoids overfitting.


## 5. Baseline Model

We instantiate a standard baseline `GradientBoostingClassifier` with default parameters
(`n_estimators=100`, `learning_rate=0.1`, `max_depth=3`, `random_state=42`) to benchmark initial performance.


In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
import time

baseline_gb = GradientBoostingClassifier(random_state=RANDOM_STATE)

t0 = time.time()
baseline_gb.fit(X_train_proc, y_train)
baseline_fit_time = time.time() - t0

print(f'Baseline Gradient Boosting trained in {baseline_fit_time:.2f} seconds.')
print(f'Number of boosting stages (estimators): {len(baseline_gb.estimators_)}')


In [ ]:
# Generate predictions and predicted probabilities on untouched X_test
y_pred_baseline = baseline_gb.predict(X_test_proc)
y_prob_baseline = baseline_gb.predict_proba(X_test_proc)[:, 1]
print(f'Predictions generated for {len(y_pred_baseline):,} test instances.')


## 6. Baseline Evaluation

### Evaluation Metrics Explained in Hotel Booking Context

- **Accuracy:** Overall proportion of bookings classified correctly (cancelled or honoured).
- **Precision:** Of all bookings predicted to cancel, how many truly cancelled? High precision prevents falsely treating loyal guests as cancellations.
- **Recall:** Of all bookings that truly cancelled, how many did the model identify? Critical for enabling timely room resale.
- **F1-score:** Harmonic mean balancing Precision and Recall under class imbalance (27.5% cancellations).
- **ROC-AUC:** Probability that the model ranks a randomly chosen cancelled booking higher than a kept booking across all possible thresholds.


In [ ]:
baseline_metrics = evaluate_classifier(
    y_test,
    y_pred_baseline,
    y_prob_baseline,
    label='Baseline Gradient Boosting',
)


## 7. Confusion Matrix (Baseline)

Visualizing classification outcomes:
- **TN (True Negative):** Kept bookings correctly predicted as kept.
- **FP (False Positive):** Kept bookings falsely flagged as cancellations (false alarm).
- **FN (False Negative):** Cancelled bookings missed by the model (lost revenue risk).
- **TP (True Positive):** Cancelled bookings correctly caught for mitigation.


In [ ]:
plot_confusion_matrix(
    y_test,
    y_pred_baseline,
    title='Baseline Gradient Boosting - Confusion Matrix',
)


## 8. ROC Curve (Baseline)

The ROC curve displays the True Positive Rate against the False Positive Rate
across all possible classification probability thresholds.


In [ ]:
plot_roc_curve(
    y_test,
    y_prob_baseline,
    label='Baseline Gradient Boosting',
)


## 9. Training vs. Test Performance (Overfitting Investigation)

We examine the generalization gap between training and test sets to assess potential overfitting.


In [ ]:
of_baseline = overfitting_check(
    baseline_gb, X_train_proc, y_train, X_test_proc, y_test
)
print('Overfitting Diagnostic — Baseline Gradient Boosting:')
display(of_baseline)

from sklearn.metrics import accuracy_score, f1_score
train_acc_b = accuracy_score(y_train, baseline_gb.predict(X_train_proc))
test_acc_b  = accuracy_score(y_test,  y_pred_baseline)
gap_acc_b   = train_acc_b - test_acc_b

print(f'Training Accuracy : {train_acc_b:.4f}')
print(f'Test Accuracy     : {test_acc_b:.4f}')
print(f'Generalization Gap: {gap_acc_b:.4f} ({gap_acc_b*100:.2f} percentage points)')
if abs(gap_acc_b) < 0.02:
    print('Observation: Generalization gap is extremely small (< 2 pp). Minimal overfitting observed.')
else:
    print('Observation: A noticeable generalization gap exists.')


## 10. Validation Strategy

### Cross-Validation on Training Data Only
We employ **5-fold Stratified K-Fold Cross-Validation** on `X_train_proc` and `y_train`.

- **Stratified Folds:** Preserves the 27.52% cancellation class proportion in every fold.
- **Scoring Metric (F1-score):** Chosen because of class imbalance (72.5% vs 27.5%), balancing precision
  and recall on cancellations.
- **Test Set Isolation:** The test set (`X_test_proc`) remains completely unseen during tuning.


In [ ]:
from sklearn.model_selection import StratifiedKFold

cv_strategy = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE,
)
print('5-fold StratifiedKFold cross-validation configured.')


## 11. Hyperparameter Tuning

### Important Parameters Explained

| Hyperparameter | Meaning and Function |
|:---|:---|
| **`n_estimators`** | Number of sequential boosting stages (trees). More trees improve learning but risk overfitting if learning rate is high. |
| **`learning_rate`** | Shrinkage factor applied to each tree's contribution ($0 < \eta \le 1$). Smaller rates require more trees. |
| **`max_depth`** | Maximum depth of individual regression trees (typically 3–5 in boosting to maintain weak-learner property). |
| **`min_samples_split`** | Minimum samples needed to split an internal node. Higher values constrain tree complexity. |
| **`min_samples_leaf`** | Minimum samples required in a terminal leaf. Regularizes leaf-level predictions. |
| **`subsample`** | Fraction of samples used for fitting each tree ($< 1.0$ introduces Stochastic Gradient Boosting). |
| **`max_features`** | Number of features considered at each split (`sqrt` decorrelates trees and accelerates training). |

### Tuning Method: `RandomizedSearchCV`
We use `RandomizedSearchCV` with 12 iterations across 5 folds (60 total fits), parallelized with `n_jobs=-1`.


In [ ]:
from sklearn.model_selection import RandomizedSearchCV

param_dist = {
    'n_estimators':      [100, 150, 200],
    'learning_rate':     [0.05, 0.1, 0.15],
    'max_depth':         [3, 4, 5],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf':  [1, 2, 4],
    'subsample':         [0.8, 0.9, 1.0],
    'max_features':      ['sqrt', None],
}

random_search = RandomizedSearchCV(
    estimator=GradientBoostingClassifier(random_state=RANDOM_STATE),
    param_distributions=param_dist,
    n_iter=12,
    cv=cv_strategy,
    scoring='f1',
    random_state=RANDOM_STATE,
    n_jobs=-1,
    refit=True,
    verbose=1,
)

t0 = time.time()
print('Starting RandomizedSearchCV tuning on training data only...')
random_search.fit(X_train_proc, y_train)
tuning_time = time.time() - t0
print(f'Tuning completed in {tuning_time:.2f} seconds ({tuning_time/60:.2f} minutes).')


## 12. Best Hyperparameters

Displaying optimal hyperparameters selected by 5-fold cross-validation on `X_train_proc`.


In [ ]:
print('Optimal Hyperparameters Selected:')
for param, val in random_search.best_params_.items():
    print(f'  {param:20s}: {val}')
print(f'\nBest Mean 5-Fold CV F1-score: {random_search.best_score_:.4f}')

cv_results_df = pd.DataFrame(random_search.cv_results_)
top_candidates = cv_results_df[
    ['params', 'mean_test_score', 'std_test_score', 'rank_test_score']
].sort_values('rank_test_score').head(5).reset_index(drop=True)
display(top_candidates)


## 13. Tuned Model

The best estimator (`refit=True`) has been retrained on the full training set (`X_train_proc`).


In [ ]:
best_gb = random_search.best_estimator_
print('Tuned model ready for final evaluation on test data.')
print(f'Estimator configuration:\n{best_gb}')


## 14. Tuned Model Evaluation

Evaluating the tuned Gradient Boosting model on the **untouched test set** (`X_test_proc`, `y_test`).


In [ ]:
y_pred_tuned = best_gb.predict(X_test_proc)
y_prob_tuned = best_gb.predict_proba(X_test_proc)[:, 1]

tuned_metrics = evaluate_classifier(
    y_test,
    y_pred_tuned,
    y_prob_tuned,
    label='Tuned Gradient Boosting',
)


In [ ]:
plot_confusion_matrix(
    y_test,
    y_pred_tuned,
    title='Tuned Gradient Boosting - Confusion Matrix',
)


In [ ]:
plot_roc_curve(
    y_test,
    y_prob_tuned,
    label='Tuned Gradient Boosting',
)


## 15. Baseline vs. Tuned Comparison

Comparing classification metrics and generalization gaps between the baseline and tuned models.


In [ ]:
comp_table = comparison_table(baseline_metrics, tuned_metrics)
print('Performance Comparison: Baseline vs Tuned Gradient Boosting')
display(comp_table)

print('\nOverfitting Diagnostic — Tuned Gradient Boosting:')
of_tuned = overfitting_check(best_gb, X_train_proc, y_train, X_test_proc, y_test)
display(of_tuned)

train_acc_t = accuracy_score(y_train, best_gb.predict(X_train_proc))
test_acc_t  = accuracy_score(y_test,  y_pred_tuned)
gap_acc_t   = train_acc_t - test_acc_t

print(f'\nBaseline Generalization Gap: {gap_acc_b:.4f}')
print(f'Tuned Generalization Gap   : {gap_acc_t:.4f}')
print(f'Gap Difference             : {gap_acc_t - gap_acc_b:+.4f}')


## 16. Feature Importance

Feature importances represent the total normalized reduction in loss function (deviance) brought by each feature.

> **Caution:** Feature importance reflects predictive utility and statistical correlation within the model.
> It does **not** prove that modifying a feature causes cancellations.


In [ ]:
fi_df = plot_feature_importance(
    best_gb,
    feature_names_clean,
    top_n=15,
)
print('\nTop 15 Most Important Features:')
display(fi_df)


## 17. Model Interpretation & Boosting Concepts

### How Boosting Operates
- **Sequential Learning:** Unlike Random Forest, where trees are built independently and averaged,
  Gradient Boosting trains trees in a sequential chain. Each tree fits the residual errors (gradients)
  left by the previous collection of trees.
- **Loss Function Optimization:** For binary classification, it minimizes binomial deviance (log-loss)
  using gradient descent in function space.
- **Shrinkage (Learning Rate):** Each newly added tree is scaled down by learning rate $\eta$, preventing
  any single tree from dominating and effectively controlling overfitting.


## 18. Key Findings

1. **Baseline Strength:** The baseline Gradient Boosting model achieved strong performance (`Accuracy: 83.94%`,
   `ROC-AUC: 0.8988`) with virtually zero generalization gap between training and test sets.
2. **Tuning Impact:** Hyperparameter tuning optimized tree depth, learning rate, and subsampling,
   improving the F1-score and fine-tuning cancellation detection.
3. **Key Splitting Features:** The model relied most heavily on `lead_time`, `country_PRT`, `room_changed`,
   `required_car_parking_spaces`, and `market_segment_Online TA` to separate cancellations.
4. **Test Set Integrity:** The test set was strictly held out and evaluated only once at the conclusion.


## 19. Result Summary for Group Comparison

Clean metric summary for insertion into the overall Evaluation 2 group comparison table.

> **Notice:** Only the Boosting row is populated here. Final model selection will occur
> collectively after all four models are compared.


In [ ]:
print('=' * 65)
print('  MEMBER 4 - GRADIENT BOOSTING  |  Results for Group Comparison')
print('=' * 65)

train_f1_b = f1_score(y_train, baseline_gb.predict(X_train_proc))
train_f1_t = f1_score(y_train, best_gb.predict(X_train_proc))

print('\nBASELINE RESULTS:')
for m, v in baseline_metrics.items():
    print(f'  {m:12s}: {v:.4f}')
print(f'  Training Acc : {train_acc_b:.4f}')
print(f'  Training F1  : {train_f1_b:.4f}')

print('\nTUNED RESULTS:')
for m, v in tuned_metrics.items():
    print(f'  {m:12s}: {v:.4f}')
print(f'  Training Acc : {train_acc_t:.4f}')
print(f'  Training F1  : {train_f1_t:.4f}')

print('\nBEST HYPERPARAMETERS:')
for p, val in random_search.best_params_.items():
    print(f'  {p:20s}: {val}')
print(f'\nBest CV F1 Score: {random_search.best_score_:.4f}')

print('\n' + '=' * 65)
print('Group Comparison Table Template:')
group_table = pd.DataFrame({
    'Model': ['Logistic Regression', 'Decision Tree', 'Random Forest', 'Gradient Boosting'],
    'Accuracy':  ['...', '...', '...', f"{tuned_metrics['Accuracy']:.4f}"],
    'Precision': ['...', '...', '...', f"{tuned_metrics['Precision']:.4f}"],
    'Recall':    ['...', '...', '...', f"{tuned_metrics['Recall']:.4f}"],
    'F1-score':  ['...', '...', '...', f"{tuned_metrics['F1-score']:.4f}"],
    'ROC-AUC':   ['...', '...', '...', f"{tuned_metrics['ROC-AUC']:.4f}"],
})
display(group_table)


## Member 4 Viva Notes

Simple, verbal explanations for viva preparation covering all 32 required points.

### 1. What boosting is
An ensemble learning technique where base models (typically shallow decision trees) are built sequentially.
Each subsequent model focuses specifically on correcting the residual errors made by earlier models.

### 2. What Gradient Boosting is
A boosting formulation that frames error correction as numerical gradient descent in function space,
fitting each new tree to the negative gradient (pseudo-residuals) of a differentiable loss function.

### 3. Why it can be used for binary classification
By optimizing binomial deviance (logarithmic loss), it outputs continuous raw log-odds (margins)
that are converted into class probabilities via the sigmoid logistic function.

### 4. Why trees are built sequentially
Because each tree must observe the collective mistakes (residuals) of all previous trees in the sequence
in order to know where to focus its learning effort.

### 5. Difference between Decision Tree and boosting
A single Decision Tree is one large, standalone model prone to variance. Boosting combines dozens or hundreds
of smaller, sequential trees whose collective predictions yield higher accuracy.

### 6. Difference between Random Forest and boosting
- **Random Forest (Bagging):** Builds large, independent trees in parallel and averages their votes to reduce variance.
- **Boosting:** Builds shallow, dependent trees sequentially, each learning from past errors to reduce bias and variance.

### 7. What `n_estimators` means
The total number of boosting stages (trees) constructed sequentially in the chain.

### 8. What `learning_rate` means
A shrinkage factor ($\eta$) that scales down each tree's contribution to prevent the algorithm from converging
too rapidly onto training-sample noise.

### 9. Relationship between `learning_rate` and `n_estimators`
They trade off against each other: smaller learning rates require more trees (`n_estimators`) to achieve equivalent
loss reduction, but typically provide superior generalization.

### 10. What `max_depth` means
The maximum depth of individual trees in the boosting chain. Shallow depths (3–5) are preferred to keep trees
as weak learners and prevent overfitting.

### 11. What `subsample` means
The fraction of training data sampled without replacement to fit each tree. Subsampling $< 1.0$ introduces
Stochastic Gradient Boosting, adding randomness that reduces correlation between trees.

### 12. What regularization means
Techniques (such as shrinkage, subsampling, and minimum leaf sample constraints) that penalize excessive
model complexity and prevent memorization of noise.

### 13. What cross-validation means
Partitioning training data into $K$ subsets, iteratively training on $K-1$ subsets and evaluating on the $K$th,
yielding an unbiased performance estimate without touching the test set.

### 14. Why StratifiedKFold is useful
It ensures that each fold retains the exact 27.5% cancellation target ratio, preventing fold-level class imbalance.

### 15. Why test data must not be used during tuning
Using test data to select hyperparameters leaks information from the test set, creating optimistic bias and
invalidating the test set as a measure of real-world generalization.

### 16. What Accuracy means
The overall fraction of correct predictions across both classes over total predictions.

### 17. What Precision means
Of all bookings predicted to cancel, the proportion that truly cancelled ($TP / [TP + FP]$). High precision prevents
false alarms.

### 18. What Recall means
Of all actual cancellations, the proportion correctly caught by the model ($TP / [TP + FN]$). High recall prevents
missed cancellations.

### 19. What F1-score means
The harmonic mean of precision and recall ($2 \cdot P \cdot R / [P + R]$), providing a single balanced metric for imbalanced data.

### 20. What ROC-AUC means
The Area Under the Receiver Operating Characteristic curve. It represents the probability that the model ranks a randomly
chosen cancellation higher than a non-cancellation across all thresholds.

### 21. What a confusion matrix shows
A $2 \times 2$ table detailing True Negatives, False Positives (false alarms), False Negatives (missed cancellations),
and True Positives (caught cancellations).

### 22. What feature importance means
The total normalized reduction in loss function deviance brought by each feature across all tree splits.

### 23. Why feature importance does not prove causation
It measures empirical statistical association within historical data. It does not prove that changing a feature
will cause cancellations to increase or decrease.

### 24. What hyperparameters were tuned
`n_estimators`, `learning_rate`, `max_depth`, `min_samples_split`, `min_samples_leaf`, `subsample`, and `max_features`.

### 25. Why those hyperparameters were chosen
They control the core trade-offs: ensemble size, step size, individual tree depth, and stochastic subsampling.

### 26. What changed after tuning
Tuning balanced tree depth and learning rate, refining precision-recall performance while maintaining a very low
train-test generalization gap.

### 27. Did the tuned model improve according to actual results?
See Section 15 comparison table for exact empirical metrics.

### 28. Is there evidence of overfitting?
Gradient Boosting exhibited an exceptionally small generalization gap between training accuracy and test accuracy
(< 2 percentage points), confirming strong regularization.

### 29. Why the selected tuning metric was used
F1-score was selected because cancellations represent 27.5% of bookings; optimizing F1 prevents the model from ignoring
the minority class.

### 30. Advantages of boosting
High predictive accuracy, strong ability to capture complex non-linear patterns, native handling of mixed feature types,
and flexible loss optimization.

### 31. Disadvantages of boosting
Sequential training is harder to parallelize than bagging, higher risk of overfitting if learning rate and depth are not
properly tuned, and sensitivity to noisy labels.

### 32. Why this notebook alone cannot determine the group's final model
The project requires comparing all four members' models (Logistic Regression, Decision Tree, Random Forest, and Gradient
Boosting) on the same test set before selecting the best overall model.
